# 01d · Video & Motion `[local]`

**Goal:** work with video: read frames, detect what moves, measure *how* it moves (optical flow), follow an object (tracking), and find faces and people with pre-deep-learning detectors.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Video as a flipbook; background = what stays still; flow = where each pixel went |
| 📐 Math | Running-average background, brightness constancy, Lucas-Kanade's 2×2 system |
| 🔨 From scratch | Running-average background subtraction, single-point Lucas-Kanade |
| 🚀 OpenCV | `VideoCapture`, MOG2/KNN, sparse + dense optical flow, CSRT/MIL trackers, Haar faces, HOG people |
| 📊 Evaluate | Synthetic videos with ground truth: foreground F1, tracking IoU, speed (FPS) |
| 🛠 Project | **Motion detector** (notebook GIF + a live webcam script) |

**About the webcam:** notebooks can't easily open live camera windows, so everything here runs on a video file. The live version is a separate script, `webcam_motion.py`, in this folder (see the end).

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image as GIF
from skimage import data as skdata
from cvlib.data import opencv_sample, download
from cvlib.viz import show, save_gif

rng = np.random.default_rng(0)

---
# 🧠 1 · Intuition

**Video = a flipbook.** A sequence of images (frames) shown quickly, typically 25–30 per second. Everything from 01a–01c works on each frame. The new ingredient is **time**: comparing frames.

**Background subtraction = "what's different from the empty scene?"** With a fixed camera, most pixels barely change. Learn what the scene normally looks like (the **background model**). Anything that differs is **foreground**: people, cars, intruders.

**Optical flow = "where did each pixel go?"** For every point, an arrow showing its movement between two frames. *Sparse* flow tracks a few good points (corners from 01c); *dense* flow gives an arrow for every pixel.

**Tracking = "follow this one thing."** You draw a box around an object once. The tracker finds it again in each new frame by searching near its last position for something that looks the same.

---
# 📐 2 · The math

### Running-average background
Keep a background image $B$ and update it slowly with each new frame $F_t$:

$$B_t = (1 - \alpha)\, B_{t-1} + \alpha\, F_t$$

$\alpha$ = learning rate. Small α (0.01) → slow to adapt, but a person who stops walking won't get absorbed into the background right away. Foreground = pixels where $|F_t - B_t| > \tau$.

**MOG2** (Mixture of Gaussians) upgrades this: each pixel keeps **several** Gaussians, because the background itself can have several normal states (leaves swaying, a flickering screen). It can also flag **shadows** (slightly darker, same color).

### Optical flow: brightness constancy
Assume a point keeps its brightness while it moves by $(u, v)$:

$$I(x + u,\, y + v,\, t + 1) = I(x, y, t)$$

A first-order Taylor expansion turns this into one equation with two unknowns:

$$I_x\, u + I_y\, v + I_t = 0$$

One equation, two unknowns: the **aperture problem** (through a small hole, you can only see motion *across* an edge, not along it).

### Lucas-Kanade
Assume all pixels in a small window move **together**. Now you have one equation per pixel, so solve by least squares:

$$\underbrace{\begin{bmatrix} \sum I_x^2 & \sum I_x I_y \\ \sum I_x I_y & \sum I_y^2 \end{bmatrix}}_{\text{the structure tensor from 01c!}} \begin{bmatrix} u \\ v \end{bmatrix} = -\begin{bmatrix} \sum I_x I_t \\ \sum I_y I_t \end{bmatrix}$$

This 2×2 matrix is solvable only if both eigenvalues are large, i.e. **at corners**. That's why sparse flow tracks Shi-Tomasi corners. For large motions, LK runs on an **image pyramid**: estimate at low resolution, then refine.

### IoU for tracking
Intersection over Union between a predicted box and the true box: 1 = perfect, 0 = no overlap. You'll use it constantly from module 06 onward.

$$\text{IoU} = \frac{\text{area}(A \cap B)}{\text{area}(A \cup B)}$$

---
# 🚀 3 · Reading video with OpenCV

In [ ]:
video_path = opencv_sample("vtest.avi")                 # ~8 MB pedestrian video from OpenCV's samples
cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"{W}×{H}, {fps:.0f} fps, {n_frames} frames ({n_frames / fps:.0f} s)")

frames = []
while len(frames) < 150:                                # first 15 s: plenty, and keeps memory small (~200 MB)
    ok, frame = cap.read()                              # ok=False at the end of the video
    if not ok:
        break
    frames.append(frame)
cap.release()                                           # always release cameras and files
print(f"loaded {len(frames)} frames, {sum(f.nbytes for f in frames) / 1e6:.0f} MB in memory")
show([frames[i] for i in (0, 70, 140)], ["frame 0", "frame 70", "frame 140"], cols=3, size=4)

The same code reads a webcam: `cv2.VideoCapture(0)` instead of a file path. To **write** video, use `cv2.VideoWriter`; in notebooks it's easier to save a GIF and display it inline, which `cvlib.viz.save_gif` does.

---
# 🔨 4 · From scratch: frame differencing and running-average background

In [ ]:
def to_gray_f(frame):
    return cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5, 5), 0).astype(np.float32)

def frame_diff_masks(frames, thresh=25):
    """Foreground = changed since the previous frame."""
    prev = to_gray_f(frames[0]); out = [np.zeros(prev.shape, np.uint8)]
    for f in frames[1:]:
        g = to_gray_f(f)
        out.append(((np.abs(g - prev) > thresh) * 255).astype(np.uint8)); prev = g
    return out

def running_avg_masks(frames, alpha=0.02, thresh=25):
    """Foreground = differs from a slowly-updated background image."""
    bg = to_gray_f(frames[0]); out = []
    for f in frames:
        g = to_gray_f(f)
        out.append(((np.abs(g - bg) > thresh) * 255).astype(np.uint8))
        bg = (1 - alpha) * bg + alpha * g             # the update rule from the math section
    return out, bg.astype(np.uint8)

fd = frame_diff_masks(frames[:150])
ra, learned_bg = running_avg_masks(frames[:150])
i = 120
show([frames[i], learned_bg, fd[i], ra[i]], ["frame 120", "learned background (people gone!)", "frame differencing", "running average"], cols=4, size=3.2)

**Frame differencing** only sees *edges* of movers (their middles overlap between frames) and loses anyone who stops. **Running average** sees whole people, because it compares against the learned empty scene.

### Lucas-Kanade for one point, from scratch

Test it on a known motion: shift an image by exactly (u, v) = (1.6, −0.9) pixels and see if we recover it.

In [ ]:
def lucas_kanade_point(I1, I2, x, y, half=7):
    """Estimate the (u, v) motion of the window around (x, y) between grayscale float images I1 → I2."""
    Ix = cv2.Sobel(I1, cv2.CV_32F, 1, 0, ksize=3) / 8          # /8 turns Sobel into a true derivative
    Iy = cv2.Sobel(I1, cv2.CV_32F, 0, 1, ksize=3) / 8
    It = I2 - I1
    win = (slice(y - half, y + half + 1), slice(x - half, x + half + 1))
    ix, iy, it = Ix[win].ravel(), Iy[win].ravel(), It[win].ravel()
    A = np.array([[ix @ ix, ix @ iy], [ix @ iy, iy @ iy]])        # structure tensor
    b = -np.array([ix @ it, iy @ it])
    return np.linalg.solve(A, b)

img1 = cv2.GaussianBlur(cv2.cvtColor(cv2.imread(str(opencv_sample("baboon.jpg"))), cv2.COLOR_BGR2GRAY), (5, 5), 0).astype(np.float32)
true_uv = np.array([1.6, -0.9])
img2 = cv2.warpAffine(img1, np.float32([[1, 0, true_uv[0]], [0, 1, true_uv[1]]]), img1.shape[::-1], borderMode=cv2.BORDER_REFLECT)

corners = cv2.goodFeaturesToTrack(img1.astype(np.uint8), maxCorners=50, qualityLevel=0.05, minDistance=20).reshape(-1, 2)
corners = corners[(corners > 20).all(1) & (corners < np.array(img1.shape[::-1]) - 20).all(1)].astype(int)

mine = np.array([lucas_kanade_point(img1, img2, x, y) for x, y in corners])
p1 = corners.astype(np.float32).reshape(-1, 1, 2)
p2, status, _ = cv2.calcOpticalFlowPyrLK(img1.astype(np.uint8), img2.astype(np.uint8), p1, None, winSize=(15, 15), maxLevel=2)
cv_uv = (p2 - p1).reshape(-1, 2)[status.ravel() == 1]

print(f"true motion            : u={true_uv[0]:.2f}, v={true_uv[1]:.2f}")
print(f"my LK (median of {len(mine)} pts) : u={np.median(mine[:, 0]):.2f}, v={np.median(mine[:, 1]):.2f}")
print(f"OpenCV pyramidal LK    : u={np.median(cv_uv[:, 0]):.2f}, v={np.median(cv_uv[:, 1]):.2f}")

Close, but not exact. A single Taylor step is approximate; OpenCV **iterates** (warp, re-solve) and uses a pyramid for larger motions. Experiment 3 shows what happens to the single-step version when the motion gets bigger.

---
# 🚀 5 · The OpenCV way

### 5.1 · MOG2 and KNN background subtractors

In [ ]:
mog2 = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=16, detectShadows=True)
knn = cv2.createBackgroundSubtractorKNN(history=200, dist2Threshold=400, detectShadows=True)
for f in frames[:121]:
    m_mog = mog2.apply(f)                                   # apply() = update the model AND return the mask
    m_knn = knn.apply(f)
show([frames[120], m_mog, m_knn, mog2.getBackgroundImage()],
     ["frame 120", "MOG2 (gray = shadow)", "KNN", "MOG2's background image"], cols=4, size=3.2)

Mask values: **255 = foreground**, **127 = shadow**, **0 = background**. To ignore shadows, keep only `mask == 255`.

### 5.2 · Sparse optical flow: tracking points over time

Detect corners once, follow them with pyramidal LK, and draw their trails. Points are re-detected when too few remain.

In [ ]:
def flow_trails(frames, n_frames=80):
    prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)
    pts = cv2.goodFeaturesToTrack(prev_gray, maxCorners=200, qualityLevel=0.01, minDistance=7)
    trails = np.zeros_like(frames[0])
    colors = rng.integers(0, 255, (1000, 3)).tolist()
    out = []
    for f in frames[1:n_frames]:
        gray = cv2.cvtColor(f, cv2.COLOR_BGR2GRAY)
        nxt, st, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, pts, None, winSize=(15, 15), maxLevel=3)
        good_new, good_old = nxt[st.ravel() == 1].reshape(-1, 2), pts[st.ravel() == 1].reshape(-1, 2)
        for k, (a, b) in enumerate(zip(good_new, good_old)):
            if np.linalg.norm(a - b) > 0.5:                     # only draw moving points
                cv2.line(trails, tuple(map(int, a)), tuple(map(int, b)), colors[k], 2)
        out.append(cv2.add(f, trails))
        prev_gray, pts = gray, good_new.reshape(-1, 1, 2)
        if len(pts) < 50:                                       # refill if we lost too many
            pts = cv2.goodFeaturesToTrack(gray, maxCorners=200, qualityLevel=0.01, minDistance=7)
    return out

trail_frames = flow_trails(frames)
show(trail_frames[-1], "sparse optical flow trails after 80 frames (only moving points drawn)", size=7)

### 5.3 · Dense optical flow (Farneback)

One motion vector per pixel. Standard visualization: **hue = direction**, **brightness = speed**.

In [ ]:
def flow_to_color(flow):
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv = np.zeros((*flow.shape[:2], 3), np.uint8)
    hsv[..., 0] = ang * 180 / np.pi / 2                         # direction → hue (0..179)
    hsv[..., 1] = 255
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)   # speed → brightness
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

a, b = (cv2.imread(str(opencv_sample(n))) for n in ("rubberwhale1.png", "rubberwhale2.png"))
flow = cv2.calcOpticalFlowFarneback(cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY), None,
                                    pyr_scale=0.5, levels=3, winsize=15, iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
g1, g2 = (cv2.cvtColor(frames[k], cv2.COLOR_BGR2GRAY) for k in (120, 121))
flow_v = cv2.calcOpticalFlowFarneback(g1, g2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
show([a, flow_to_color(flow), frames[121], flow_to_color(flow_v)],
     ["rubberwhale frame 1", "dense flow (hue = direction)", "vtest frame 121", "dense flow: walkers light up"], cols=2, size=5)

Dense flow is what video models, video stabilization and frame interpolation are built on. Deep models (RAFT and successors) compute it far more accurately; you'll meet them after the transformer modules.

### 5.4 · Object tracking: CSRT and MIL

Initialize with a box once, then call `update(frame)` every frame.

| Tracker | Idea | Speed | Robustness |
|---|---|---|---|
| **CSRT** | correlation filter + spatial reliability map | medium | good, handles shape changes |
| **MIL** | online-trained classifier on many patches | slow | ok, drifts |
| *(deep trackers: Nano, ViT)* | small neural nets | fast on GPU | best: covered after module 08 |

In [ ]:
init_box = (638, 238, 50, 85)                                # x, y, w, h of the walker on the right, frame 0

def run_tracker(make, frames, box, n=150):
    tr = make(); tr.init(frames[0], box)
    boxes, t0 = [box], time.perf_counter()
    for f in frames[1:n]:
        ok, b = tr.update(f)
        boxes.append(tuple(int(v) for v in b) if ok else None)
    return boxes, (n - 1) / (time.perf_counter() - t0)

track = {}
for name, make in [("CSRT", cv2.TrackerCSRT_create), ("MIL", cv2.TrackerMIL_create)]:
    track[name], speed = run_tracker(make, frames, init_box)
    lost = sum(b is None for b in track[name])
    print(f"{name}: {speed:6.0f} FPS | reported 'lost' in {lost} of {len(track[name])} frames")

panels = []
for k in (0, 50, 100, 149):
    v = frames[k].copy()
    for name, color in [("CSRT", (0, 255, 0)), ("MIL", (0, 0, 255))]:
        b = track[name][k]
        if b is not None:
            cv2.rectangle(v, (b[0], b[1]), (b[0] + b[2], b[1] + b[3]), color, 2)
    panels.append(v)
show(panels, [f"frame {k}  (green = CSRT, red = MIL)" for k in (0, 50, 100, 149)], cols=2, size=5)

⚠️ **Look closely before trusting "reported lost in 0 frames".** Around frame 40 the walker passes behind the lamp post, right next to other people. Both trackers quietly **jump to a different person** (an *identity switch*) and keep going with full confidence.

This is the core weakness of classic single-object trackers: **they drift silently**. Two lessons:
- Never judge a tracker by whether it *says* it's lost. Measure it against ground truth (section 6).
- Real systems **re-detect** objects every few frames and link detections over time (*tracking-by-detection*, after module 06).

### 5.5 · Classic detectors: Haar faces and HOG people

Before deep learning (pre-2014), these were *the* detectors in cameras and phones:
- **Haar cascade** (Viola-Jones, 2001): thousands of simple light/dark rectangle tests, arranged in a cascade so most non-face windows are rejected after a few tests. Very fast, but only frontal faces, and many false alarms.
- **HOG + linear SVM** (Dalal-Triggs, 2005): describe a window by its **histogram of gradient directions** (01b's gradients again!), then classify person / not-person.

Both scan windows at many positions and scales: the "sliding window" idea that deep detectors (module 06) replaced.

OpenCV 5's pip package no longer bundles the cascade files, so we download one.

In [ ]:
cascade_path = download("https://raw.githubusercontent.com/opencv/opencv/4.x/data/haarcascades/haarcascade_frontalface_default.xml", subdir="models")
face_cascade = cv2.CascadeClassifier(str(cascade_path))

def detect_faces(img, scale=1.1, neighbors=5):
    return face_cascade.detectMultiScale(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), scaleFactor=scale, minNeighbors=neighbors, minSize=(30, 30))

astro = cv2.cvtColor(skdata.astronaut(), cv2.COLOR_RGB2BGR)
messi = cv2.imread(str(opencv_sample("messi5.jpg")))
hog = cv2.HOGDescriptor()
hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())

panels, titles = [], []
for name, im in [("astronaut", astro), ("messi5", messi)]:
    v = im.copy()
    for (x, y, w_, h_) in detect_faces(im):
        cv2.rectangle(v, (x, y), (x + w_, y + h_), (0, 255, 0), 2)
    panels.append(v); titles.append(f"Haar faces: {name}")
people, _ = hog.detectMultiScale(frames[100], winStride=(8, 8), padding=(8, 8), scale=1.05)
v = frames[100].copy()
for (x, y, w_, h_) in people:
    cv2.rectangle(v, (x, y), (x + w_, y + h_), (255, 0, 0), 2)
panels.append(v); titles.append(f"HOG people: {len(people)} found")
show(panels, titles, cols=3, size=4)

---
# 🛠 Project · Motion detector

Background subtraction → clean the mask with morphology (01b) → contours → boxes around anything big enough.

In [ ]:
def motion_detector(frames, min_area=400, history=200, var_threshold=16):
    sub = cv2.createBackgroundSubtractorMOG2(history=history, varThreshold=var_threshold, detectShadows=True)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    out, counts = [], []
    for f in frames:
        mask = sub.apply(f)
        mask = np.where(mask == 255, 255, 0).astype(np.uint8)              # drop shadows (127)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)              # remove specks
        mask = cv2.dilate(mask, kernel, iterations=2)                      # join body parts
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        v = f.copy(); n = 0
        for c in contours:
            if cv2.contourArea(c) < min_area:
                continue
            x, y, w_, h_ = cv2.boundingRect(c); n += 1
            cv2.rectangle(v, (x, y), (x + w_, y + h_), (0, 255, 0), 2)
        cv2.putText(v, f"moving objects: {n}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2, cv2.LINE_AA)
        out.append(v); counts.append(n)
    return out, counts

det_frames, counts = motion_detector(frames)
gif_path = save_gif(det_frames[30:150:2], env.out_dir / "motion_detector.gif", fps=10)
print(f"saved {gif_path}")
GIF(filename=str(gif_path))

In [ ]:
plt.figure(figsize=(10, 2.8))
plt.plot(counts); plt.xlabel("frame"); plt.ylabel("objects detected")
plt.title("Motion detector output over time (the first frames = background still being learned)")
plt.grid(alpha=0.3); plt.show()

### Live version: your webcam

Run this from the project root in a terminal (a window opens; press **q** to quit):

```bash
uv run python notebooks/01_classic_cv/webcam_motion.py
```

macOS will ask for camera permission for your terminal app the first time.

---
# 📊 6 · Evaluate

Real videos rarely come with labels, so we build **synthetic videos with exact ground truth**: textured objects moving over a textured background with noise and a slow lighting change.

In [ ]:
def make_synthetic_video(n=120, size=(360, 480), seed=0):
    r_ = np.random.default_rng(seed)
    H_, W_ = size
    bg = cv2.resize(cv2.imread(str(opencv_sample("building.jpg"))), (W_, H_))
    sprite = cv2.resize(cv2.imread(str(opencv_sample("baboon.jpg"))), (50, 50))
    objs = [dict(x=30.0, y=60.0, vx=3.2, vy=1.1), dict(x=400.0, y=250.0, vx=-2.5, vy=-0.8), dict(x=200.0, y=20.0, vx=0.6, vy=2.4)]
    frames_, masks_, track_boxes = [], [], []
    for t in range(n):
        f = bg.astype(np.float32) * (1.0 + 0.15 * np.sin(t / 25))           # slow lighting change
        m = np.zeros((H_, W_), np.uint8)
        for k, o in enumerate(objs):
            x, y = int(o["x"] + o["vx"] * t), int(o["y"] + o["vy"] * t)
            x, y = np.clip(x, 0, W_ - 50), np.clip(y, 0, H_ - 50)
            f[y:y + 50, x:x + 50] = sprite
            m[y:y + 50, x:x + 50] = 255
            if k == 0:
                track_boxes.append((x, y, 50, 50))                          # ground truth for tracking object 0
        f = np.clip(f + r_.normal(0, 6, f.shape), 0, 255).astype(np.uint8)
        frames_.append(f); masks_.append(m)
    return frames_, masks_, track_boxes

syn_frames, syn_masks, syn_boxes = make_synthetic_video()
show([syn_frames[0], syn_frames[60], syn_masks[60]], ["synthetic frame 0", "frame 60", "ground-truth mask, frame 60"], cols=3, size=3.5)

### 6.1 · Background subtraction: pixel precision, recall, F1

- **Precision**: of the pixels we flagged as moving, how many really were?
- **Recall**: of the pixels that really moved, how many did we flag?
- **F1**: their harmonic mean (high only when both are high).

These three numbers come back in every module from here on.

In [ ]:
def prf(pred, truth):
    p, t = pred > 0, truth > 0
    tp, fp, fn = (p & t).sum(), (p & ~t).sum(), (~p & t).sum()
    prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
    return prec, rec, 2 * prec * rec / max(prec + rec, 1e-9)

def eval_method(masks_pred, skip=30):                           # skip the warm-up frames
    scores = np.array([prf(p, t) for p, t in zip(masks_pred[skip:], syn_masks[skip:])])
    return scores.mean(0)

def opencv_masks(sub, frames_):
    return [np.where(sub.apply(f) == 255, 255, 0).astype(np.uint8) for f in frames_]

methods = {}
t0 = time.perf_counter(); methods["frame differencing"] = (frame_diff_masks(syn_frames), time.perf_counter() - t0)
t0 = time.perf_counter(); methods["running average"] = (running_avg_masks(syn_frames)[0], time.perf_counter() - t0)
t0 = time.perf_counter(); methods["MOG2"] = (opencv_masks(cv2.createBackgroundSubtractorMOG2(200, 16, True), syn_frames), time.perf_counter() - t0)
t0 = time.perf_counter(); methods["KNN"] = (opencv_masks(cv2.createBackgroundSubtractorKNN(200, 400, True), syn_frames), time.perf_counter() - t0)

print(f"{'method':<20}{'precision':>10}{'recall':>9}{'F1':>7}{'FPS':>9}")
for name, (masks_pred, dt) in methods.items():
    p, r, f1 = eval_method(masks_pred)
    print(f"{name:<20}{p:>10.2f}{r:>9.2f}{f1:>7.2f}{len(syn_frames) / dt:>9.0f}")

### 6.2 · Tracking: IoU against the true box

In [ ]:
def iou(a, b):
    if a is None or b is None:
        return 0.0
    ax2, ay2, bx2, by2 = a[0] + a[2], a[1] + a[3], b[0] + b[2], b[1] + b[3]
    iw = max(0, min(ax2, bx2) - max(a[0], b[0])); ih = max(0, min(ay2, by2) - max(a[1], b[1]))
    inter = iw * ih
    return inter / (a[2] * a[3] + b[2] * b[3] - inter)

plt.figure(figsize=(10, 3))
for name, make in [("CSRT", cv2.TrackerCSRT_create), ("MIL", cv2.TrackerMIL_create)]:
    boxes, speed = run_tracker(make, syn_frames, syn_boxes[0], n=len(syn_frames))
    ious = [iou(b, t) for b, t in zip(boxes, syn_boxes)]
    print(f"{name}: mean IoU {np.mean(ious):.2f} | frames with IoU > 0.5: {np.mean(np.array(ious) > 0.5):.0%} | {speed:.0f} FPS")
    plt.plot(ious, label=name)
plt.xlabel("frame"); plt.ylabel("IoU with truth"); plt.ylim(0, 1.05); plt.legend(); plt.grid(alpha=0.3)
plt.title("Tracking accuracy over time (objects cross paths mid-video)"); plt.show()

### 6.3 · Face detector false alarms

A detector is only useful if it doesn't fire on everything. Run it on 5 images with **no human faces** and see how `minNeighbors` (how many overlapping detections a face needs) trades missed faces for false alarms.

In [ ]:
no_faces = {name: cv2.imread(str(opencv_sample(name))) for name in ["fruits.jpg", "starry_night.jpg", "stuff.jpg", "baboon.jpg"]}
no_faces["chelsea (cat)"] = cv2.cvtColor(skdata.chelsea(), cv2.COLOR_RGB2BGR)
print(f"{'minNeighbors':>12} | {'boxes on 2 photos with 1 face each':>34} | {'false alarms on 5 face-free images':>34}")
for nb in [1, 3, 5, 10]:
    boxes_on_faces = len(detect_faces(astro, neighbors=nb)) + len(detect_faces(messi, neighbors=nb))
    false = sum(len(detect_faces(im, neighbors=nb)) for im in no_faces.values())
    print(f"{nb:>12} | {boxes_on_faces:>34} | {false:>34}")

Low `minNeighbors` → more boxes everywhere: duplicates and false alarms on the face photos (6 boxes for 2 faces), plus false faces in paintings and fruit. At 5, it's exactly 2 boxes and 0 false alarms, which is why 5 is a common default. This **precision vs. recall trade-off** via a threshold is the same one you'll tune for every detector in this course.

---
# 🧪 7 · Experiments: predict first, then run

**Experiment 1: background learning rate.** A car parks in the scene at frame 10 and stays. With α = 0.001, 0.02 and 0.2, predict: how many frames until the running-average background "forgets" it's there?

In [ ]:
# Prediction: ...
scene = cv2.resize(cv2.imread(str(opencv_sample("building.jpg"))), (320, 240))
parked = []
for t in range(300):
    f = scene.copy()
    if t >= 10:
        cv2.rectangle(f, (120, 140), (200, 190), (40, 40, 200), -1)          # the "car" arrives at frame 10
    parked.append(f)
car = (slice(140, 191), slice(120, 201))
for alpha in [0.001, 0.02, 0.2]:
    m, _ = running_avg_masks(parked, alpha=alpha)
    still_seen = [(mk[car] > 0).mean() for mk in m[10:]]
    gone = next((k for k, v in enumerate(still_seen) if v < 0.5), None)
    print(f"alpha={alpha:<5}: car absorbed into background after {gone if gone is not None else '> 290'} frames")
# Large α adapts fast: good for lighting changes, but anything that stops moving vanishes quickly.
# Small α remembers the empty scene: parked objects stay visible, but it reacts slowly to real scene changes.
# Choose α from what should count as "background" in your application.

**Experiment 2: MOG2 `varThreshold`.** Lower = more sensitive. Predict the effect on precision and recall on the synthetic video.

In [ ]:
# Prediction: ...
for vt in [4, 16, 64]:
    p, r, f1 = eval_method(opencv_masks(cv2.createBackgroundSubtractorMOG2(200, vt, True), syn_frames))
    print(f"varThreshold={vt:>2}: precision {p:.2f} | recall {r:.2f} | F1 {f1:.2f}")

**Experiment 3: single-step Lucas-Kanade vs. large motion.** Your from-scratch LK uses one Taylor step on one level. Predict what happens as the true shift grows to 3 and 8 pixels.

In [ ]:
# Prediction: ...
for shift in [1.0, 3.0, 8.0]:
    img2s = cv2.warpAffine(img1, np.float32([[1, 0, shift], [0, 1, 0]]), img1.shape[::-1], borderMode=cv2.BORDER_REFLECT)
    est = np.median([lucas_kanade_point(img1, img2s, x, y)[0] for x, y in corners])
    p2s, sts, _ = cv2.calcOpticalFlowPyrLK(img1.astype(np.uint8), img2s.astype(np.uint8), p1, None, winSize=(15, 15), maxLevel=3)
    cv_est = np.median((p2s - p1).reshape(-1, 2)[sts.ravel() == 1, 0])
    print(f"true u = {shift:4.1f} | my single-step LK: {est:5.2f} | OpenCV pyramidal LK: {cv_est:5.2f}")
# The Taylor approximation only holds for small motions (~1–2 px). The pyramid shrinks big motions
# into small ones at low resolution: the key trick behind practical optical flow.

---
# 📝 8 · Recap

**Cheat sheet**
- `cv2.VideoCapture(path_or_0)` → `cap.read()` in a loop → `cap.release()`. Webcam = index 0.
- **Frame differencing** sees only edges of movers. **Running average** learns the empty scene. **MOG2/KNN** model each pixel with several "normal" states and flag shadows (127).
- Clean masks with opening + dilation, then contours → boxes: the classic motion detector.
- **Optical flow** assumes brightness constancy: $I_x u + I_y v + I_t = 0$. **Lucas-Kanade** solves a 2×2 system per window, solvable at corners. Pyramids handle large motion.
- Sparse flow (LK on corners) for tracking points; dense flow (Farneback) for every pixel.
- **Trackers**: CSRT = accurate and medium speed; MIL = slow, drifts. Deep trackers come later.
- **Haar** (faces) and **HOG+SVM** (people) = classic sliding-window detectors. Fast, but brittle.
- **Evaluate with synthetic ground truth**: pixel precision/recall/F1 for masks, IoU for boxes, FPS for speed.

**Interview-style questions**
1. Why does frame differencing miss the middle of a moving person?
2. A background subtractor slowly "forgets" a parked car. Which parameter controls this, and what's the trade-off?
3. Derive the optical flow constraint equation. Why is it not enough on its own?
4. Why does Lucas-Kanade work at corners but fail on flat regions and along edges?
5. How do image pyramids help optical flow with large motions?
6. Define IoU. A tracker has mean IoU 0.4: is that good?
7. Why did deep detectors replace Haar cascades and HOG+SVM?

---
# 🎉 Module 01 complete

You can now load, clean, filter, threshold, measure, match, align, stitch, calibrate, and track using classic CV alone. Many production systems still use exactly these tools around (or instead of) a neural network.

**Next:** `02_nn_basics/02a_numpy_network.ipynb`, where you'll build a neural network from scratch.